# 🤖 Agentic AI with Bigdata.com Search API

This notebook demonstrates how your AI agents can interact with **Bigdata.com Search API** to access real-time market intelligence, news, filings, and earnings transcripts—combined with your internal data sources.

## What This Demonstrates

**Bigdata.com Integration Patterns:**
- **Knowledge Graph API** → Resolve company names to entity IDs for precise filtering
- **Search API** → Query news, SEC filings, earnings transcripts with sentiment scores
- **Tool-based Architecture** → Wrap APIs as callable tools for any agentic framework

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine external market data with proprietary insights

**Framework Flexibility:**
> This demo uses **LangChain** and **LangSmith** for observability, but the integration pattern applies to any agentic framework: **CrewAI**, **AutoGen**, **Google A2A**, or custom implementations. The key is wrapping Bigdata.com APIs as tools your agents can call.

---

## Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         LangChain ReAct Agent                               │
│                    (Reasoning + Acting Pattern)                             │
└─────────────┬─────────────────┬─────────────────┬───────────────────────────┘
              │                 │                 │
              ▼                 ▼                 ▼
    ┌─────────────────┐ ┌─────────────┐ ┌─────────────────────────────────┐
    │  Internal DB    │ │   FAISS     │ │   Bigdata.com APIs              │
    │  (SQLite)       │ │  Vector     │ │   ┌───────────────────────────┐ │
    │  ✓ Portfolios   │ │  Store      │ │   │ Knowledge Graph API       │ │
    │  ✓ Holdings     │ │  ✓ Research │ │   │ ✓ Entity lookup           │ │
    │  ✓ Transactions │ │    Docs     │ │   ├───────────────────────────┤ │
    │                 │ │             │ │   │ Search API                │ │
    │                 │ │             │ │   │ ✓ News & filings          │ │
    │                 │ │             │ │   │ ✓ Transcripts             │ │
    │                 │ │             │ │   │ ✓ Sentiment analysis      │ │
    └─────────────────┘ └─────────────┘ │   └───────────────────────────┘ │
           LOCAL             LOCAL      └─────────────────────────────────┘
                                                    EXTERNAL
```

**Key Benefits:**
- **Single agent interface** for internal DB, vector store, and Bigdata.com Search/Knowledge Graph
- **Production-ready** tooling: retry, logging, KG entity cache (see `langgraph_core.py`)
- **Observability** via LangSmith for tool selection and latency
- **Flexibility**: Same pattern works with CrewAI, AutoGen, or custom frameworks—wrap APIs as tools

---

## 📊 LangSmith Tracing

![LangSmith Tracing](./static/langsmith_search.png)

---

## 1️⃣ Install Dependencies

> **Note:** If using `uv` (recommended), skip this cell. Run `uv pip install -r requirements.txt` instead.

In [1]:
# Dependencies are installed via uv: uv pip install -r requirements.txt
# %pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q

## 2️⃣ Import Libraries

Import core utilities and display helpers from `langgraph_core`.

In [2]:
import os
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML

# LangChain
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI

# Local utilities - environment, data sources, and display helpers
import sys
sys.path.append('.')
from langgraph_core import (
    # Environment & Data Setup
    setup_environment,
    create_financial_database,
    create_vector_store,
    
    # Tool Providers
    get_bigdata_tools,
    get_database_tools,
    get_vectorstore_tools,
    
    # Display Utilities
    display_query,
    display_response,
    display_tools_used,
    display_citations
)

# Load environment variables
load_dotenv()

print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [3]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project='bigdata-agent-demo',
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")
print("\n🔗 View traces at: https://smith.langchain.com")

✅ LangSmith tracing enabled → Project: bigdata-agent-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, TSM, PLTR)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)

🔗 View traces at: https://smith.langchain.com


## 4️⃣ Load Local Tools

Load tools that interact with local data sources.

In [4]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for tool in local_db_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for tool in local_vector_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transaction...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity...


## 5️⃣ Load Bigdata.com Tools

Load external tools for market intelligence:

- **Knowledge Graph API** - Entity lookup and company identification
- **Search API** - News, filings, transcripts with sentiment

In [5]:
# Get Bigdata.com tools
bigdata_tools = get_bigdata_tools()
print(f"✅ Loaded {len(bigdata_tools)} Bigdata.com tools:")
for tool in bigdata_tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Loaded 2 Bigdata.com tools:
   - bigdata_lookup_company: Look up a company's Bigdata entity ID using the Knowledge Gr...
   - bigdata_search_news: Search Bigdata.com for financial news and market intelligenc...


## 6️⃣ Combine All Tools

Merge tools from all sources for the agent.

In [6]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata.com tools: {len(bigdata_tools)}")


✅ Total tools available: 5
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata.com tools: 2


## 7️⃣ Create LangChain Agent

Create an agent with all tools configured.

**ReAct Pattern:**
- **Reasoning**: Plans which tools to use based on query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

In [7]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com APIs):**
1. `bigdata_lookup_company` - Look up company entity IDs from the Knowledge Graph
2. `bigdata_search_news` - Search financial news and market intelligence with source citations

**Internal Data (Company Systems):**
3. `internal_query_database` - Execute SQL queries on portfolio/transaction database
4. `internal_portfolio_summary` - Get portfolio holdings and performance summary
5. `internal_search_research` - Search internal investment research documents

Guidelines:
- For company news, first use `bigdata_lookup_company` to get the entity_id, then use it in `bigdata_search_news`
- For portfolio questions, use `internal_portfolio_summary` or `internal_query_database` with SQL
- Combine external market data with internal holdings/research for comprehensive analysis

**Source Attribution & Citation Format:**
- **CRITICAL:** Add inline citations with superscript numbers [1], [2], [3] immediately after each factual claim
- Each distinct factual claim should have its own citation(s) - typically 1-2 sources per claim
- Place citations right after the claim, not at the end of sentences: "NVIDIA's revenue grew 409% [1][2] driven by AI chip demand [3]"
- When multiple sources support the SAME specific claim, group them: [1][2]
- When sources support DIFFERENT claims in one sentence, distribute them appropriately: "AMD is challenging [1][2], while Intel focuses on foundry [3]"
- **DO NOT bunch citations** - avoid patterns like [1][2][3][4][5] at sentence end. Break compound sentences if needed
- Use sequential numbering [1], [2], [3]... throughout your response
- Limit to 3-4 citations per individual factual claim (split into sub-claims if needed)
- For internal data: Briefly mention "From internal database" or "According to internal research"
- DO NOT include a "Sources:" list at the end of your response - sources are displayed separately

**Example Good Citation:**
"Competition from AMD and Intel is intensifying [1][2], while tech giants like Amazon and Google are developing custom AI chips [3][4]."

**Example Bad Citation (avoid):**
"Competition from AMD, Intel, and custom chips by companies like Amazon and Google is intensifying [1][2][3][4][5]."
Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create agent using langchain.agents.create_agent
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: gpt-4o")
print(f"   System prompt configured")
print("\n🔍 Agent Tool Selection:")
print("   • Portfolio/holdings → internal_query_database")
print("   • Internal research → internal_search_research")
print("   • Company lookup → bigdata_lookup_company")
print("   • Market news → bigdata_search_news")

✅ Agent created with 5 tools
   Model: gpt-4o
   System prompt configured

🔍 Agent Tool Selection:
   • Portfolio/holdings → internal_query_database
   • Internal research → internal_search_research
   • Company lookup → bigdata_lookup_company
   • Market news → bigdata_search_news


## 8️⃣ System Prompt

The agent uses the `SYSTEM_PROMPT` defined above. Key elements:

**External Tools (Bigdata.com):**
- `bigdata_lookup_company` - Get entity IDs for filtering news
- `bigdata_search_news` - Search financial news with citations

**Internal Tools:**
- `internal_query_database` - SQL queries on portfolios
- `internal_portfolio_summary` - Holdings & performance
- `internal_search_research` - Semantic search on research docs

**Guidelines:**
- Look up entity_id first, then search news
- Use internal tools for portfolio questions
- Combine sources for comprehensive analysis
- Use inline citations [1], [2] for external data

---

## 9️⃣ Example Queries

Run queries that combine data from multiple sources.

### Query 1: Multi-Source NVIDIA Analysis

Combines internal holdings, research, and external news.

In [8]:
query = """I need a comprehensive analysis of NVIDIA:
1. What's our current position in NVIDIA across all portfolios?
2. What does our internal research say about NVIDIA's investment thesis?
3. What's the latest news about NVIDIA from market sources?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
display_citations(result)

### NVIDIA Position in Portfolios

Currently, there are no holdings of NVIDIA in any of the portfolios according to the internal database query results.

### Internal Research on NVIDIA

According to our internal research, NVIDIA remains a top pick in the semiconductor space. Key highlights from the latest investment thesis include:

1. **Data Center Revenue**: NVIDIA's data center revenue reached $18.4 billion, marking a 409% year-over-year increase, driven by the demand for H100/H200 GPUs used in AI training.
2. **Next-Gen Architecture**: The upcoming Blackwell Architecture, with B100/B200 GPUs, is expected to launch in Q2 2025, offering 2.5 times the performance of current models.
3. **Software Ecosystem**: NVIDIA's CUDA ecosystem, with over 4 million developers, creates significant switching costs, reinforcing its competitive advantage.
4. **AI Inference Market**: The total addressable market for AI inference is projected to reach $150 billion by 2027 as enterprises scale AI deployments.

Risk factors include potential export restrictions to China, competition from AMD, and supply constraints. The price target is set at $950, with a "STRONG BUY" rating [From internal research].

### Latest Market News on NVIDIA

1. **Investment in CoreWeave**: NVIDIA has invested an additional $2 billion in CoreWeave Inc. to expand AI computing capacity by over 5 gigawatts by 2030 [1].
2. **Collaboration with DeepSeek**: NVIDIA provided technical support to DeepSeek, a Chinese startup, to enhance its AI model, despite US export controls [2].
3. **Market Performance**: NVIDIA's shares have soared 38.8% over the past year, highlighting its strong market performance alongside Micron Technology [3].
4. **Microsoft's AI Chip**: Microsoft is developing its AI chip to reduce reliance on NVIDIA hardware, indicating competitive pressures in the AI chip market [4].

These insights provide a comprehensive view of NVIDIA's current market position, internal strategic outlook, and recent developments.

### Query 2: Portfolio Risk Assessment

Analyzes portfolio holdings with internal research and external news.

In [9]:
query = """Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. Are there any recent news events affecting these holdings?"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
display_citations(result)

### Current Holdings and Performance of AI & Semiconductor Focus Portfolio (PF002)

1. **NVIDIA Corporation (NVDA)**
   - Shares: 12,000
   - Average Cost: $450.00
   - Current Price: $875.50
   - Market Value: $10,506,000
   - Unrealized P&L: $5,106,000

2. **Broadcom Inc. (AVGO)**
   - Shares: 1,500
   - Average Cost: $850.00
   - Current Price: $1,425.00
   - Market Value: $2,137,500
   - Unrealized P&L: $862,500

3. **Palantir Technologies (PLTR)**
   - Shares: 25,000
   - Average Cost: $18.50
   - Current Price: $65.25
   - Market Value: $1,631,250
   - Unrealized P&L: $1,168,750

4. **Advanced Micro Devices (AMD)**
   - Shares: 8,000
   - Average Cost: $95.00
   - Current Price: $145.25
   - Market Value: $1,162,000
   - Unrealized P&L: $402,000

5. **Taiwan Semiconductor (TSM)**
   - Shares: 3,000
   - Average Cost: $110.00
   - Current Price: $185.75
   - Market Value: $557,250
   - Unrealized P&L: $227,250

Total Market Value: $15,994,000  
Total Unrealized P&L: $7,766,500  
(From internal database)

### Risks Identified in Internal Research

1. **Valuation Risk**: High valuation multiples in the sector, with companies trading at over 30x forward P/E. The AI premium could compress if monetization does not meet expectations.
2. **Regulatory Risk**: Potential impacts from antitrust actions and digital market regulations, particularly affecting companies like Google and Apple.
3. **China Exposure**: Export controls could affect revenue, especially for companies like NVIDIA, which has significant revenue exposure to China.
4. **AI Bubble Risk**: Concerns about infrastructure spending outpacing actual demand and unproven ROI on enterprise AI investments.
(According to internal research)

### Recent News Events Affecting Holdings

1. **NVIDIA (NVDA)**
   - Invested $2 billion in CoreWeave to expand AI computing capacity[1].
   - Involved in a controversy over providing technical support to a Chinese startup despite US export controls[2].

2. **Broadcom (AVGO)**
   - Strengthened its position in the AI compute and infrastructure ecosystem, receiving positive analyst ratings[3].
   - Stock performance has been mixed, with recent dips despite broader market gains[4].

3. **Palantir (PLTR)**
   - Partnered with Innodata to accelerate AI initiatives, boosting its stock[5].
   - Anticipates high double-digit growth, with a significant stock price increase over the past three years[6].

4. **AMD (AMD)**
   - Collaborating with Gigabyte to enhance AI capabilities in gaming and PC products[7].
   - Expected to report strong Q4 earnings, with significant year-over-year growth[8].

5. **TSMC (TSM)**
   - Announced major investments in the US and UAE, reshaping global AI chip supply[9].
   - Reported strong earnings and margin expansion, with positive analyst outlooks[10].

These developments highlight both opportunities and challenges within the AI and semiconductor sectors, reflecting the dynamic nature of the market.

### Query 3: News Sentiment Analysis

Focuses on external market intelligence with sentiment.

## 🔟 Key Benefits of This Architecture

1. **Unified data access** — One agent queries SQLite, FAISS, and Bigdata.com Search/Knowledge Graph via tools.
2. **Production-ready** — Retry and logging for Bigdata APIs; KG entity cache by ticker/company name.
3. **Citation support** — Inline citations [1], [2] and source display for Search results.
4. **Observability** — LangSmith traces show tool calls, latency, and reasoning.
5. **Reusability** — All setup and tools live in `langgraph_core.py`; notebooks stay simple.

## 🎯 Next Steps

- **Add Research Agent** — Use `create_hierarchical_agent(include_research_agent=True)` for deep research with citations (see `agent_to_research_agent.ipynb`).
- **Custom tools** — Implement `@tool` functions and add them via `get_bigdata_tools()` or a custom list before `create_agent()`.
- **Different backends** — Point `setup_environment(db_path=...)` at your own DB; pass custom documents to `create_vector_store(documents=...)`.
- **LangSmith** — Set `LANGSMITH_API_KEY` and tune prompts/tools using trace data.

---

## 📚 Additional Resources

- **Bigdata.com API docs**: https://docs.bigdata.com
- **LangGraph / LangChain**: https://langchain-ai.github.io/langgraph/
- **LangSmith**: https://smith.langchain.com
- **Agent_To_BigData README**: `README.md` in this folder

**Questions?** support@bigdata.com

In [10]:
query = """What's the market sentiment around Apple in the last 30 days?
Look for news about:
1. Product launches and innovations
2. Financial performance
3. Regulatory issues

Also check if we hold Apple in any portfolios."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
display_citations(result)

### Market Sentiment Around Apple in the Last 30 Days

#### Product Launches and Innovations
1. **Acquisitions and Product Plans**: Apple recently acquired the Israeli startup Q.ai, which is expected to enhance its AI capabilities[1]. The company is also planning to launch over 20 new products this year, including updates to iPhones, iPads, Macs, and Apple Watches[2].
2. **AI and Siri Developments**: There is anticipation around a new Siri launch that will leverage AI advancements, although challenges in AI commercialization remain[3].

#### Financial Performance
1. **Earnings Expectations**: Analysts are optimistic about Apple's financial performance, expecting strong iPhone demand and a positive AI strategy to drive growth. The company is set to report earnings with an expected revenue growth of 10% to 12% for the quarter[4][5].
2. **Stock Performance**: Despite being the second-worst-performing Dow Jones stock so far in 2026, analysts see potential buying opportunities due to Apple's strong fundamentals and AI prospects[6].

#### Regulatory Issues
1. **Fines and Scrutiny**: Apple has faced significant regulatory challenges, particularly in Europe, where it was fined $851 million last year for antitrust and privacy violations[7][8]. The company continues to face scrutiny over its App Store payment practices[9].

### Portfolio Holdings
Apple is held in two of our portfolios:
- **PF001 (US Large Cap Growth)**: 15,000 shares with a market value of $2,778,750 and an unrealized profit of $641,250.
- **PF003 (Diversified Tech Leaders)**: 25,000 shares with a market value of $4,631,250 and an unrealized profit of $756,250.

These holdings represent a significant portion of each portfolio, indicating a strong belief in Apple's long-term potential (From internal database).

### Query 4: Portfolio Strategy Review

Combines internal strategy memos with current market conditions.

In [11]:
query = """Review our Q1 2025 portfolio strategy:
1. What allocation changes did our internal research recommend?
2. What are our top holdings by market value?
3. Any recent news that might affect our strategy?

Provide a summary of whether we should stay the course or adjust."""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display results
display_tools_used(result)
display_response(result)
display_citations(result)

### Portfolio Strategy Review for Q1 2025

#### 1. Recommended Allocation Changes
According to our internal research, the following allocation changes were recommended for Q1 2025:
- **Increase**: 
  - NVIDIA (NVDA): +3% weight due to AI training demand exceeding supply.
  - Meta Platforms (META): +2% weight as it is undervalued relative to its AI investments.
  - Palantir Technologies (PLTR): +1% weight due to accelerating government AI contracts.
- **Maintain**:
  - Microsoft (MSFT): Current weight due to balanced growth and value.
  - Apple (AAPL): Current weight as services growth offsets hardware challenges.
- **Reduce**:
  - Advanced Micro Devices (AMD): -1% weight due to valuation concerns versus execution risk.
  - Salesforce (CRM): -1% weight due to uncertainty in Agentforce adoption[1].

#### 2. Top Holdings by Market Value
- **US Large Cap Growth (PF001)**:
  - Microsoft (MSFT): $3,324,000
  - Apple (AAPL): $2,778,750
  - Meta Platforms (META): $2,632,500

- **AI & Semiconductor Focus (PF002)**:
  - NVIDIA (NVDA): $10,506,000
  - Broadcom (AVGO): $2,137,500
  - Palantir Technologies (PLTR): $1,631,250

- **Diversified Tech Leaders (PF003)**:
  - NVIDIA (NVDA): $7,004,000
  - Microsoft (MSFT): $6,232,500
  - Apple (AAPL): $4,631,250

#### 3. Recent News Impacting Strategy
Recent news about NVIDIA includes:
- NVIDIA's shares have soared 38.8% last year, indicating strong performance and potential upside[2].
- NVIDIA invested an additional $2 billion in CoreWeave to expand AI computing capacity, which could enhance its market position[3].
- However, NVIDIA's involvement with DeepSeek, a Chinese AI model, despite US export controls, could pose regulatory risks[4].

### Summary and Recommendation
Given the strong performance and strategic investments by NVIDIA, along with the recommended increases in allocation for NVDA, META, and PLTR, it seems prudent to stay the course with the current strategy. However, attention should be paid to regulatory risks and the potential impact of new AI chip developments by competitors like Microsoft, which could affect NVIDIA's market share[5]. Adjustments should be considered if these risks materialize or if there are significant changes in market conditions.

---

## 🔟 Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**

| Query Type | Tool Used |
|------------|----------|
| Portfolio positions | `internal_query_database` |
| Internal research | `internal_search_research` |
| Company lookup | `bigdata_lookup_company` |
| Market news | `bigdata_search_news` |

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

---

## 🎯 Next Steps

**Extend this architecture:**

1. **Add More External Data Sources**
   ```python
   # Example: Add earnings calendar tool
   from langgraph_core import get_research_agent_tool
   research_tools = get_research_agent_tool()
   all_tools += research_tools
   ```

2. **Implement Response Caching**
   ```python
   from functools import lru_cache
   import hashlib
   
   def cache_key(query: str) -> str:
       return hashlib.md5(query.encode()).hexdigest()
   ```

3. **Add Rate Limiting**
   ```python
   from ratelimit import limits, sleep_and_retry
   
   @sleep_and_retry
   @limits(calls=10, period=60)
   def rate_limited_search(query: str):
       return bigdata_search_news(query)
   ```

4. **Add Error Handling & Retries**
   ```python
   from tenacity import retry, stop_after_attempt, wait_exponential
   
   @retry(stop=stop_after_attempt(3), wait=wait_exponential())
   def robust_api_call(query: str):
       return agent.invoke({"messages": [("user", query)]})
   ```

**Production Considerations:**
- Implement proper error handling for API failures
- Add rate limiting to manage API costs
- Cache frequently requested data
- Monitor performance in LangSmith

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **Search API Reference**: https://docs.bigdata.com/search-api
- **Knowledge Graph API**: https://docs.bigdata.com/knowledge-graph
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com